<a href="https://colab.research.google.com/github/NagaRaju1991/google_colab_notebooks/blob/fsds_course/01_Introduction_perceptron_pytorch_tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Lab 1: PyTorch Fundamentals
# This notebook covers the essentials of PyTorch: Tensors, Math, and Autograd.


What is CUDA? </br>
CUDA is NVIDIA's parallel computing platform and programming model that allows developers to use NVIDIA GPUs for general-purpose computing (not just graphics).

✅ Full Form: Compute Unified Device Architecture </br>
✅ Created: NVIDIA (2006) </br>
✅ Purpose: GPGPU (General-Purpose GPU computing) </br>
✅ Now: Often used without expanding acronym </br>


In [ ]:
import torch
import numpy as np

In [ ]:
torch.cuda.is_available()

False

In [ ]:
### Exercise 1: Tensor Creation
### Tensors are the primary data structure in PyTorch, similar to NumPy arrays but with GPU support.

# 1. Create a 2x3 tensor of zeros
zeros = torch.zeros(2, 3)

# 2. Create a tensor from a Python list
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)

# 3. Create a random tensor with shape (3, 3)
rand_tensor = torch.randn(3, 3)

print(f"Zeros Tensor:\n{zeros}")
print(f"Random Tensor:\n{rand_tensor}")
print(f"Shape: {rand_tensor.shape} | Device: {rand_tensor.device}")

Zeros Tensor:
tensor([[0., 0., 0.],
        [0., 0., 0.]])
Random Tensor:
tensor([[-0.4062, -2.8564, -0.9213],
        [-0.8941, -1.7107, -0.2089],
        [ 0.8526,  0.9117, -0.7230]])
Shape: torch.Size([3, 3]) | Device: cpu


In [ ]:
# ## Exercise 2: Tensor Math
# Understanding the difference between element-wise operations and matrix multiplication.

tensor = torch.ones(3, 3) * 2

# 1. Element-wise multiplication
element_wise = tensor * tensor

# 2. Matrix multiplication (Dot product)
# Use @ or torch.matmul
matmul = tensor @ tensor

print(f"Original:\n{tensor}")
print(f"Element-wise (2*2):\n{element_wise}")
print(f"Matrix Mult (Row x Col):\n{matmul}")

Original:
tensor([[2., 2., 2.],
        [2., 2., 2.],
        [2., 2., 2.]])
Element-wise (2*2):
tensor([[4., 4., 4.],
        [4., 4., 4.],
        [4., 4., 4.]])
Matrix Mult (Row x Col):
tensor([[12., 12., 12.],
        [12., 12., 12.],
        [12., 12., 12.]])


In [ ]:
# ## Exercise 3: NumPy Bridge
# PyTorch and NumPy share underlying memory when on CPU, making conversion nearly instant.

# PyTorch to NumPy
t = torch.ones(5)
n = t.numpy()

# NumPy to PyTorch
n = np.array([1, 2, 3])
t = torch.from_numpy(n)

print(f"NumPy Array: {n}")
print(f"Converted Tensor: {t}")

NumPy Array: [1 2 3]
Converted Tensor: tensor([1, 2, 3])


####  PyTorch Autograd Explained - Complete Guide

#### What is Autograd? 🤖

**Autograd** = **Automatic Differentiation Engine** in PyTorch.

**Purpose**: Automatically computes **gradients** (derivatives) for **any computation** you define using Tensors.

**Why needed?** Neural networks require **backpropagation** → compute ∂Loss/∂Weights for every parameter → **millions of derivatives!**

**Autograd solves**: "You do forward pass → Autograd does backward pass AUTOMATICALLY!"

---

##  How Autograd Works (Dynamic Computation Graph)

### 1. **Forward Pass**: Build the Graph
```python
import torch

x = torch.tensor(2.0, requires_grad=True)  # Leaf node (track gradients!)
y = x ** 2                                   # Node: MulBackward0
z = 3 * y + 2                               # Node: AddBackward0

print(z)  # tensor(14., grad_fn=<AddBackward0>)
# ∂z/∂x = ∂z/∂y × ∂y/∂x = (3) × (2x) = 3 × 4 = 12 ✓

In [ ]:
import torch

x = torch.tensor(2.0, requires_grad=True)  # Leaf node (track gradients!)
y = x ** 2                                   # Node: MulBackward0
z = 3 * y + 2                               # Node: AddBackward0

print(z)

z

print(y)
print(z.grad_fn)
#print(y.grad_fn.next_functions)
#print(z.grad_fn)
#print(z.grad_fn.next_functions[0][0].next_functions)

tensor(14., grad_fn=<AddBackward0>)
tensor(4., grad_fn=<PowBackward0>)


In [ ]:
# ## Exercise 4: Autograd (Automatic Differentiation)
# This is the most important part of PyTorch. It tracks operations to calculate gradients.

# 1. Define a tensor with requires_grad=True
x = torch.tensor([3.0], requires_grad=True)

# 2. Define a function y = x^2 + 2x + 1
y = x**2 + 2*x + 1

# 3. Compute gradients (Backpropagation)
y.backward()

# 4. Check the gradient (dy/dx = 2x + 2)
# Since x = 3, dy/dx should be 2(3) + 2 = 8


print(f"Input x: {x.item()}")
print(f"Equation result y: {y.item()}")
print(f"Calculated Gradient (dy/dx): {x.grad.item()}")

Input x: 3.0
Equation result y: 16.0
Calculated Gradient (dy/dx): 8.0


### .backward() traverses graph backwards → fills .grad of leaf nodes
z.backward() → traverses graph backwards → fills .grad of leaf nodes


| Line               | Purpose         | Result              |
| ------------------ | --------------- | ------------------- |
| requires_grad=True | Enable tracking | Builds graph        |
| loss.backward()    | Backprop        | w.grad = 12.0       |
| torch.no_grad()    | Update safely   | No graph during GD  |
| w.grad.zero_()     | Reset           | Ready for next iter |

In [ ]:
import torch

# 1. Create parameters (weights) to optimize
w = torch.tensor(2.0, requires_grad=True)  # Initial weight
target = 1.0

# 2. Forward pass (build graph)
pred = w ** 2
loss = (pred - target) ** 2  # MSE loss

print(f"Loss: {loss.item()}")  # tensor(9., grad_fn=<PowBackward0>)

# 3. Backward pass (compute gradients)
loss.backward()

'''
  ∂Loss/∂w = ∂Loss/∂pred × ∂pred/∂w
         = 2(pred-target) × 2w
         = 2(4-1) × 2(2)
         = 2(3) × 4
         = 6 × 4
         = 24

'''

print(f"w.grad: {w.grad}")  # ∂Loss/∂w = 24.0 (should decrease w!)

# 4. Update weights (Gradient Descent)
with torch.no_grad():  # Disable autograd during update
    w -= 0.1 * w.grad   # lr=0.1
    w.grad.zero_()      # Reset gradients

print(f"New w: {w.item()}")  # w closer to 1.0


Loss: 9.0
w.grad: 24.0
New w: -0.40000009536743164


In [ ]:
# ## Exercise 5: Linear Regression (The "Manual" Way)
# Let's try to find the weight 'w' for the equation y = w * x

# Training Data
X = torch.tensor([1, 2, 3, 4], dtype=torch.float32)
Y = torch.tensor([2, 4, 6, 8], dtype=torch.float32) # y = 2x

# Initialize weight randomly
w = torch.tensor([0.0], dtype=torch.float32, requires_grad=True)

# Training Loop
learning_rate = 0.01

for epoch in range(20):
    # 1. Forward Pass (Prediction)
    y_pred = X * w

    # 2. Loss (Mean Squared Error)
    loss = ((y_pred - Y)**2).mean()

    # 3. Backward Pass
    loss.backward()

    # 4. Update Weights
    with torch.no_grad():
        w -= learning_rate * w.grad

    # 5. Zero Gradients for next loop
    w.grad.zero_()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}: w = {w.item():.3f}, loss = {loss.item():.3f}")

print(f"\nFinal Prediction for x=5: {5 * w.item():.3f} (Expected 10.0)")

Epoch 0: w = 0.300, loss = 30.000
Epoch 5: w = 1.246, loss = 5.906
Epoch 10: w = 1.665, loss = 1.163
Epoch 15: w = 1.851, loss = 0.229

Final Prediction for x=5: 9.612 (Expected 10.0)


#### Lab: TensorFlow & Keras Fundamentals
##### This notebook covers the essentials of TensorFlow: Tensors, GradientTape, and Keras layers.


In [ ]:
import tensorflow as tf
import numpy as np

In [ ]:
# ## Exercise 1: Constant vs Variable Tensors
# In TensorFlow, `tf.constant` is immutable, while `tf.Variable` allows for updates (weights).

# 1. Create a constant tensor
const_tensor = tf.constant([[1.0, 2.0], [3.0, 4.0]])

# 2. Create a variable (usually used for model weights)
var_tensor = tf.Variable([2.0, 5.0])

# 3. Create a tensor of zeros with shape (2, 5)
zeros = tf.zeros([2, 5])

print(f"Constant:\n{const_tensor}")
print(f"Variable:\n{var_tensor}")
print(f"Zeros Shape: {zeros.shape}")

Constant:
[[1. 2.]
 [3. 4.]]
Variable:
<tf.Variable 'Variable:0' shape=(2,) dtype=float32, numpy=array([2., 5.], dtype=float32)>
Zeros Shape: (2, 5)


- Exercise 2

In [ ]:
# ## Exercise 2: Tensor Math
# TensorFlow uses specific functions for math, though standard operators (+, *, @) are overloaded.

a = tf.constant([[1, 2], [3, 4]], dtype=tf.float32)
b = tf.constant([[5, 6], [7, 8]], dtype=tf.float32)

# 1. Element-wise addition
add = a + b

# 2. Matrix Multiplication (Dot Product)
# Use tf.matmul or the @ operator
matmul = a @ b

print(f"Matrix Sum:\n{add}")
print(f"Matrix Product:\n{matmul}")

Matrix Sum:
[[ 6.  8.]
 [10. 12.]]
Matrix Product:
[[19. 22.]
 [43. 50.]]


- Exercise 3

In [ ]:
# ## Exercise 3: Automatic Differentiation (GradientTape)
# TensorFlow uses the `tf.GradientTape` context manager to record operations for gradients.

x = tf.Variable(3.0)

with tf.GradientTape() as tape:
    # Define function y = x^2 + 2x + 1
    y = x**2 + 2*x + 1

# Calculate dy/dx
dy_dx = tape.gradient(y, x)

print(f"Input x: {x.numpy()}")
print(f"Gradient (2x + 2): {dy_dx.numpy()}") # Expected: 2(3) + 2 = 8

Input x: 3.0
Gradient (2x + 2): 8.0


In [ ]:
# =============================================================================
# TensorFlow GradientTape - LINE-BY-LINE EXECUTION
# =============================================================================

import tensorflow as tf  # TensorFlow library

# =============================================================================
# STEP 1: CREATE TRAINABLE VARIABLE
# =============================================================================
# tf.Variable(value) → Creates mutable tensor that GradientTape TRACKS automatically
x = tf.Variable(3.0)
# - Value: 3.0 (initial weight/parameter)
# - AUTOMATICALLY watched by GradientTape (no manual watch() needed)
# - Purpose: Parameters we want gradients FOR (weights, biases)

print(f"x: {x.numpy()}")  # x: 3.0 ✓

# =============================================================================
# STEP 2: GRADIENTTAPE CONTEXT (RECORD OPERATIONS)
# =============================================================================
# with tf.GradientTape() as tape:
# - tape = "recording device" for TensorFlow operations
# - Records ALL ops on Variables inside context → builds computation graph
# - Dynamic graph (built on-the-fly, destroyed after use)

with tf.GradientTape() as tape:  # tape = empty recording tape
    # Define function: y = x² + 2x + 1 (quadratic)
    y = x**2 + 2*x + 1
    # Forward pass executed:
    # x=3 → x²=9 → 2x=6 → y=9+6+1=16
    # Graph recorded: x ──→ [Square] ──→ [Mul(2)] ──→ [Add(1)] ──→ y

# INSIDE tape: y = 16.0 (scalar tensor)

# =============================================================================
# STEP 3: COMPUTE GRADIENT (BACKWARD PASS)
# =============================================================================
# tape.gradient(target, sources)
# - target=y: Output whose gradient we want
# - sources=x: Input variable w.r.t. which we differentiate
# - Returns: ∂y/∂x (gradient tensor)

dy_dx = tape.gradient(y, x)
# Mathematical verification:
# y = x² + 2x + 1
# ∂y/∂x = 2x + 2
# At x=3: 2(3) + 2 = 8

# =============================================================================
# STEP 4: DISPLAY RESULTS
# =============================================================================
print(f"Input x: {x.numpy()}")              # Input x: 3.0
print(f"Gradient (2x + 2): {dy_dx.numpy()}") # Gradient: 8.0

# =============================================================================
# COMPLETE EXECUTION TRACE
# =============================================================================
"""
1. x = Variable(3.0)           # Trackable parameter
2. tape records: y = x²+2x+1   # y=16.0, graph built
3. dy_dx = ∂y/∂x = 2x+2=8.0    # Chain rule applied
4. Print results ✓
"""


x: 3.0
Input x: 3.0
Gradient (2x + 2): 8.0


'\n1. x = Variable(3.0)           # Trackable parameter\n2. tape records: y = x²+2x+1   # y=16.0, graph built\n3. dy_dx = ∂y/∂x = 2x+2=8.0    # Chain rule applied\n4. Print results ✓\n'

In [ ]:
import tensorflow as tf
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]